## Sentiment Analysis and Model Validation with API Deployment
This notebook analyzes sentiment using **RoBERTa** and **VADER models** while providing an API interface via Flask.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import torch
import logging
from datetime import datetime
from flask import Flask, request, jsonify
import os

# Logging configuration
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

### Step 1: Load Dataset with Error Handling

In [ ]:
# Configurable file path with error handling
def load_dataset(file_path):
    """
    Load dataset from the specified file path and limit it to the first 500 rows.
    Ensures file existence and handles errors during file reading.
    
    Args:
        file_path (str): Path to the CSV file containing data.

    Returns:
        pd.DataFrame: Loaded dataframe containing data.
    """
    try:
        if not os.path.isfile(file_path):
            raise FileNotFoundError(f"File {file_path} does not exist.")
        df = pd.read_csv(file_path).head(500)
        logger.info(f"Successfully loaded dataset from {file_path}")
        return df
    except Exception as e:
        logger.error(f"Error loading dataset: {e}")
        raise

# Load dataset
DATA_FILE = "Reviews.csv"
df = load_dataset(DATA_FILE)

### Step 2: Define Sentiment Analysis Functions

In [ ]:
def get_roberta_sentiment(text, tokenizer, model):
    """
    Perform sentiment analysis using the RoBERTa model.

    Args:
        text (str): Input text for sentiment analysis.
        tokenizer: Tokenizer for the RoBERTa model.
        model: Pre-trained RoBERTa model.

    Returns:
        str: Predicted sentiment (Negative, Neutral, Positive).
    """
    try:
        encoded_text = tokenizer(text, return_tensors='pt', max_length=512, truncation=True)
        with torch.no_grad():
            output = model(**encoded_text)
        scores = softmax(output[0][0].numpy())
        return ['Negative', 'Neutral', 'Positive'][np.argmax(scores)]
    except Exception as e:
        logger.error(f"Error during RoBERTa sentiment analysis: {e}")
        return 'Neutral'  # Default to Neutral on error

def get_vader_sentiment(text, sia):
    """
    Perform sentiment analysis using VADER (rule-based sentiment analysis).
    
    Args:
        text (str): Input text for sentiment analysis.
        sia: VADER SentimentIntensityAnalyzer instance.
    
    Returns:
        str: Predicted sentiment (Negative, Neutral, Positive).
    """
    scores = sia.polarity_scores(text)
    if scores['compound'] >= 0.05:
        return 'Positive'
    elif scores['compound'] <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

### Step 3: Initialize Pre-Trained Models

In [ ]:
# Initialize models
MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL)
    sia = SentimentIntensityAnalyzer()
    logger.info("Successfully loaded models")
except Exception as e:
    logger.error(f"Error loading models: {e}")
    raise

### Step 4: Evaluate Sentiments on Dataset

In [ ]:
def evaluate_models(df):
    roberta_predictions = []
    vader_predictions = []
    for text in df['Text']:
        roberta_predictions.append(get_roberta_sentiment(text, tokenizer, model))
        vader_predictions.append(get_vader_sentiment(text, sia))
    df['RoBERTa_Prediction'] = roberta_predictions
    df['VADER_Prediction'] = vader_predictions
    return df

df = evaluate_models(df)

### Step 5: Validate Model Performance

In [ ]:
y_true = df['Sentiment']  # True labeled data
y_pred_roberta = df['RoBERTa_Prediction']
y_pred_vader = df['VADER_Prediction']

print("\nRoBERTa Performance:")
print(classification_report(y_true, y_pred_roberta))

print("\nVADER Performance:")
print(classification_report(y_true, y_pred_vader))

### Step 6: Save Results to CSV

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f'sentiment_model_comparison_{timestamp}.csv'
df.to_csv(output_file, index=False)
print(f"Validation complete. Results saved to {output_file}.")

### Step 7: Deploy API with Flask

In [ ]:
app = Flask(__name__)

@app.route("/analyze", methods=["POST"])
def analyze_sentiment():
    data = request.get_json()
    text = data.get("text", "")
    if not text:
        return jsonify({"error": "No text provided"}), 400
    roberta_result = get_roberta_sentiment(text, tokenizer, model)
    vader_result = get_vader_sentiment(text, sia)
    return jsonify({
        "input_text": text,
        "roberta_sentiment": roberta_result,
        "vader_sentiment": vader_result
    })

app.run(debug=True)